# Power-Off 180 Ground Track Simulator

Interactive visualization of a power-off 180 emergency approach for a **Piper Warrior III** flying left traffic to **Runway 29**.

## Aircraft Parameters
- **Glide airspeed**: 73 KIAS
- **Pattern altitude**: 1000 ft AGL
- **Downwind distance**: 0.5 NM (3000 ft) from centerline

## How to Use
- Adjust the **headwind/tailwind** slider to see how wind along the runway affects ground speed and turn timing
- Adjust the **crosswind** slider to see wind drift and required crab corrections
- Watch how **turn radii** change with groundspeed
- Notice when to start the base turn for optimal runway alignment

## How It Works

### Assumptions
- Aircraft maintains best glide speed (73 KIAS) throughout the maneuver
- Constant 25-degree bank angle in all turns
- Sea level conditions (TAS = IAS)
- Left traffic pattern with standard 90-degree turns to base and final
- Abeam the touchdown point at 1000 ft AGL

### Key Concept: Glide Ratio in the Air Mass

The Piper Warrior III has a **glide ratio of 11.3:1** — meaning in still air, you travel 11.3 feet forward for each foot of altitude lost. However, this is measured **relative to the air mass**, not the ground.

Wind changes your groundspeed, which changes how much ground you cover per foot of altitude:

**Effective glide ratio over ground** = (Air mass L/D) × (Groundspeed / TAS)

For example, with an 11.3:1 glide ratio at 73 kt TAS:
- No wind: 11.3:1 (cover 11.3 ft ground per 1 ft altitude)
- 20 kt headwind (53 kt GS): 11.3 × 53/73 = **8.2:1** (much worse)
- 20 kt tailwind (93 kt GS): 11.3 × 93/73 = **14.4:1** (much better)

### Why Headwind Means Turn Base Earlier

With a headwind on final approach:
1. Your groundspeed is slower
2. Your effective glide ratio is worse (you lose altitude faster relative to the ground)
3. You need **more altitude** to cover the same final approach distance
4. Therefore, you must turn base **earlier** (closer to abeam) to preserve that altitude

The opposite is true with a tailwind — you glide farther over the ground, so you can fly a longer downwind leg.

### Equations Used

| Quantity | Formula |
|----------|---------|
| Effective L/D | L/D × (groundspeed / TAS) |
| Altitude loss | ground distance / effective L/D |
| Turn radius | V² / (g × tan(bank)) |
| Arc length | radius × angle (in radians) |

### How the Simulator Solves for the Pattern

The simulator finds the downwind leg length that results in exactly 1000 ft of altitude loss from abeam to touchdown:

1. **Calculate groundspeeds** for each leg based on wind
2. **Calculate effective glide ratios** for each leg
3. **Use binary search** to find the downwind distance where total altitude loss = 1000 ft
4. **Build the ground track** from 5 segments: downwind → base turn → base leg → final turn → final approach

In [ ]:
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

# Aircraft and pattern constants
GLIDE_AIRSPEED = 73       # KIAS (best glide)
TAS = 73                  # True airspeed at sea level (kt), TAS ≈ IAS
BANK_ANGLE = 25           # degrees (fixed)
TOUCHDOWN_Y = 1000        # ft down runway (aiming point)
PATTERN_ALT = 1000        # ft AGL at abeam

# Piper Warrior III glide performance
# This is L/D in the AIR MASS, not over ground
# Effective ground ratio = GLIDE_RATIO * (groundspeed / TAS)
GLIDE_RATIO = 11.3     # 11:3 still-air glide ratio, 8.1-9 effective with flaps

# Physics constants
KNOTS_TO_FPS = 1.68781
G = 32.174

In [ ]:
def calculate_crab_angle(airspeed_knots, crosswind_knots):
    """Calculate wind correction angle (crab) needed to maintain track."""
    if abs(crosswind_knots) >= airspeed_knots:
        return np.sign(crosswind_knots) * 45
    return np.degrees(np.arcsin(crosswind_knots / airspeed_knots))


def calculate_turn_radius(groundspeed_knots, bank_angle_deg):
    """Calculate turn radius: r = V^2 / (g * tan(bank))"""
    v_fps = groundspeed_knots * KNOTS_TO_FPS
    bank_rad = np.radians(bank_angle_deg)
    if np.tan(bank_rad) == 0:
        return float('inf')
    return (v_fps ** 2) / (G * np.tan(bank_rad))


def compute_arc_points(cx, cy, radius, start_deg, end_deg, num_points=30):
    """Compute arc points. 0 deg = +X axis, CCW positive."""
    angles = np.linspace(np.radians(start_deg), np.radians(end_deg), num_points)
    return cx + radius * np.cos(angles), cy + radius * np.sin(angles)


def compute_arc_length(radius, angle_deg):
    """Compute arc length for a given radius and angle in degrees."""
    return radius * np.radians(abs(angle_deg))


def compute_power_off_180_track(headwind_knots, crosswind_knots, downwind_distance):
    """
    Compute ground track for power-off 180 approach.

    KEY PHYSICS:
    - Glide ratio (11.3:1) is in the AIR MASS, not over ground
    - Wind changes groundspeed, which changes EFFECTIVE ground glide ratio
    - Effective_ratio = GLIDE_RATIO * (groundspeed / TAS)
    - Altitude_loss = ground_distance / effective_ratio

    CRITICAL INSIGHT:
    With headwind on final, you cover LESS ground per foot of altitude lost.
    This means you need MORE altitude for final → turn base EARLIER.
    """
    # ========== GROUNDSPEEDS FOR EACH LEG ==========
    gs_downwind = max(GLIDE_AIRSPEED + headwind_knots, 20)  # tailwind on downwind
    gs_base = max(GLIDE_AIRSPEED + crosswind_knots, 20)     # XW effect on base
    gs_final = max(GLIDE_AIRSPEED - headwind_knots, 20)     # headwind on final

    # ========== EFFECTIVE GLIDE RATIOS ==========
    eff_ratio_downwind = GLIDE_RATIO * gs_downwind / TAS
    eff_ratio_base = GLIDE_RATIO * gs_base / TAS
    eff_ratio_final = GLIDE_RATIO * gs_final / TAS
    eff_ratio_base_turn = (eff_ratio_downwind + eff_ratio_base) / 2
    eff_ratio_final_turn = (eff_ratio_base + eff_ratio_final) / 2

    # Crab angles
    crab_downwind = calculate_crab_angle(GLIDE_AIRSPEED, crosswind_knots)
    crab_base = calculate_crab_angle(GLIDE_AIRSPEED, headwind_knots)
    crab_final = calculate_crab_angle(GLIDE_AIRSPEED, -crosswind_knots)

    # Turn radii (based on groundspeed during turn)
    R_base = calculate_turn_radius((gs_downwind + gs_base) / 2, BANK_ANGLE)
    R_final = calculate_turn_radius((gs_base + gs_final) / 2, BANK_ANGLE)

    # Turn arc lengths
    base_turn_arc = compute_arc_length(R_base, 90)
    final_turn_arc = compute_arc_length(R_final, 90)

    # ========== BINARY SEARCH FOR DOWNWIND LEG ==========
    # Find downwind_leg that gives total altitude loss = PATTERN_ALT
    lo, hi = 100.0, 8000.0
    
    for _ in range(30):
        mid = (lo + hi) / 2.0
        
        # Compute geometry for this downwind leg
        base_entry_y = -mid
        base_center_x = downwind_distance + R_base
        base_center_y = base_entry_y
        base_exit_x = base_center_x
        base_exit_y = base_center_y - R_base
        
        final_center_x = -R_final
        final_center_y = base_exit_y + R_final
        rollout_y = final_center_y
        final_leg_length = TOUCHDOWN_Y - rollout_y
        base_leg_length = abs(base_exit_x - (-R_final))
        
        # Compute altitude losses
        alt_loss_downwind = mid / eff_ratio_downwind
        alt_loss_base_turn = base_turn_arc / eff_ratio_base_turn
        alt_loss_base = base_leg_length / eff_ratio_base
        alt_loss_final_turn = final_turn_arc / eff_ratio_final_turn
        alt_loss_final = final_leg_length / eff_ratio_final
        
        total_alt = alt_loss_downwind + alt_loss_base_turn + alt_loss_base + alt_loss_final_turn + alt_loss_final
        
        if abs(total_alt - PATTERN_ALT) < 1.0:
            break
        elif total_alt > PATTERN_ALT:
            hi = mid
        else:
            lo = mid
    
    downwind_leg = mid

    # ========== FINAL GEOMETRY ==========
    base_entry_y = -downwind_leg
    base_center_x = downwind_distance + R_base
    base_center_y = base_entry_y
    base_exit_x = base_center_x
    base_exit_y = base_center_y - R_base

    final_center_x = -R_final
    final_center_y = base_exit_y + R_final
    rollout_y = final_center_y
    final_leg_length = TOUCHDOWN_Y - rollout_y
    base_leg_length = abs(base_exit_x - (-R_final))

    # Recalculate altitude losses for reporting
    alt_loss_downwind = downwind_leg / eff_ratio_downwind
    alt_loss_base_turn = base_turn_arc / eff_ratio_base_turn
    alt_loss_base = base_leg_length / eff_ratio_base
    alt_loss_final_turn = final_turn_arc / eff_ratio_final_turn
    alt_loss_final = final_leg_length / eff_ratio_final
    alt_fixed = alt_loss_base_turn + alt_loss_base + alt_loss_final_turn + alt_loss_final

    # ========== BUILD TRACK ==========
    track_x, track_y = [], []

    # 1. Downwind: from abeam (Y=0) to base turn entry
    dw_x = np.full(20, float(downwind_distance))
    dw_y = np.linspace(0, base_entry_y, 20)
    track_x.extend(dw_x)
    track_y.extend(dw_y)

    # 2. Base turn: 180 to 270 deg
    bt_x, bt_y = compute_arc_points(base_center_x, base_center_y, R_base, 180, 270, 25)
    track_x.extend(bt_x)
    track_y.extend(bt_y)

    # 3. Base leg
    bl_x = np.linspace(base_exit_x, -R_final, 15)
    bl_y = np.full(15, base_exit_y)
    track_x.extend(bl_x)
    track_y.extend(bl_y)

    # 4. Final turn: 270 to 360 deg
    ft_x, ft_y = compute_arc_points(final_center_x, final_center_y, R_final, 270, 360, 25)
    track_x.extend(ft_x)
    track_y.extend(ft_y)

    # 5. Final: from rollout to touchdown
    fa_x = np.zeros(20)
    fa_y = np.linspace(rollout_y, TOUCHDOWN_Y, 20)
    track_x.extend(fa_x)
    track_y.extend(fa_y)

    track_x_arr = np.array(track_x)
    track_y_arr = np.array(track_y)

    return {
        'track_x': track_x_arr,
        'track_y': track_y_arr,
        'base_turn_point': (downwind_distance, base_entry_y),
        'final_turn_point': (-R_final, base_exit_y),
        'threshold_point': (0, 0),  # Runway threshold at Y=0
        'ground_speeds': {'downwind': gs_downwind, 'base': gs_base, 'final': gs_final},
        'effective_ratios': {'downwind': eff_ratio_downwind, 'base': eff_ratio_base, 'final': eff_ratio_final},
        'crab_angles': {'downwind': crab_downwind, 'base': crab_base, 'final': crab_final},
        'turn_radii': {'base': R_base, 'final': R_final},
        'dist_past_abeam': abs(base_entry_y),
        'altitude_budget': {
            'total': PATTERN_ALT,
            'downwind': alt_loss_downwind,
            'base_turn': alt_loss_base_turn,
            'base': alt_loss_base,
            'final_turn': alt_loss_final_turn,
            'final': alt_loss_final,
            'fixed_total': alt_fixed
        },
        'leg_distances': {
            'downwind': downwind_leg,
            'base_turn': base_turn_arc,
            'base': base_leg_length,
            'final_turn': final_turn_arc,
            'final': final_leg_length
        }
    }

In [ ]:
# Create widgets
headwind_slider = widgets.FloatSlider(
    value=0, min=-20, max=20, step=1,
    description='Headwind (kts):',
    style={'description_width': '140px'},
    layout=widgets.Layout(width='400px'),
    continuous_update=True
)

crosswind_slider = widgets.FloatSlider(
    value=0, min=-20, max=20, step=1,
    description='Crosswind (kts):',
    style={'description_width': '140px'},
    layout=widgets.Layout(width='400px'),
    continuous_update=True
)

downwind_slider = widgets.FloatSlider(
    value=4500, min=3500, max=5500, step=100,
    description='Downwind (ft):',
    style={'description_width': '140px'},
    layout=widgets.Layout(width='400px'),
    continuous_update=True
)

metrics_html = widgets.HTML(value='')

# Compute initial track (downwind_distance is negative for left of runway)
initial_data = compute_power_off_180_track(0, 0, -downwind_slider.value)


def compute_altitude_profile(data):
    """
    Compute altitude along track using segment-by-segment effective glide ratios.
    """
    track_x = data['track_x']
    track_y = data['track_y']
    n_pts = len(track_x)
    
    dx = np.diff(track_x)
    dy = np.diff(track_y)
    segment_dist = np.sqrt(dx**2 + dy**2)
    cum_dist = np.concatenate([[0], np.cumsum(segment_dist)])
    
    # Leg boundaries (approximate indices)
    downwind_end = 19
    base_turn_end = 44
    base_leg_end = 59
    final_turn_end = 84
    
    eff_ratios = data['effective_ratios']
    eff_ratio_base_turn = (eff_ratios['downwind'] + eff_ratios['base']) / 2
    eff_ratio_final_turn = (eff_ratios['base'] + eff_ratios['final']) / 2
    
    altitude = np.zeros(n_pts)
    altitude[0] = PATTERN_ALT
    
    for i in range(1, n_pts):
        dist = segment_dist[i-1]
        
        if i <= downwind_end:
            eff_ratio = eff_ratios['downwind']
        elif i <= base_turn_end:
            eff_ratio = eff_ratio_base_turn
        elif i <= base_leg_end:
            eff_ratio = eff_ratios['base']
        elif i <= final_turn_end:
            eff_ratio = eff_ratio_final_turn
        else:
            eff_ratio = eff_ratios['final']
        
        alt_loss = dist / eff_ratio
        altitude[i] = max(0, altitude[i-1] - alt_loss)
    
    return cum_dist, altitude


# Create ground track figure
fig = go.FigureWidget()

# Runway
fig.add_trace(go.Scatter(
    x=[-75, -75, 75, 75, -75], y=[0, 3000, 3000, 0, 0],
    fill='toself', fillcolor='#4a4a4a',
    line=dict(color='#4a4a4a', width=1),
    name='Runway', hoverinfo='skip'
))

# Centerline
fig.add_trace(go.Scatter(
    x=[0, 0], y=[0, 3000],
    mode='lines', line=dict(color='white', width=2, dash='dash'),
    name='Centerline', hoverinfo='skip'
))

# Aiming point
fig.add_trace(go.Scatter(
    x=[-50, -50, 50, 50, -50], y=[900, 1100, 1100, 900, 900],
    fill='toself', fillcolor='rgba(46, 204, 113, 0.5)',
    line=dict(color='#2ecc71', width=2),
    name='Aiming Point', hoverinfo='skip'
))

# Threshold markers
fig.add_trace(go.Scatter(
    x=[-60, -60, -40, -40, -60], y=[50, 200, 200, 50, 50],
    fill='toself', fillcolor='white', line=dict(color='white', width=1),
    name='Threshold', hoverinfo='skip', showlegend=False
))
fig.add_trace(go.Scatter(
    x=[40, 40, 60, 60, 40], y=[50, 200, 200, 50, 50],
    fill='toself', fillcolor='white', line=dict(color='white', width=1),
    hoverinfo='skip', showlegend=False
))

# Abeam marker (trace 5)
fig.add_trace(go.Scatter(
    x=[-downwind_slider.value], y=[0],
    mode='markers+text',
    marker=dict(size=15, color='#3498db', symbol='circle'),
    text=['Abeam'], textposition='top center',
    textfont=dict(size=12, color='#3498db'),
    name='Abeam Point', hoverinfo='skip'
))

# Ground track (trace 6)
fig.add_trace(go.Scatter(
    x=initial_data['track_x'], y=initial_data['track_y'],
    mode='lines', line=dict(color='#8e44ad', width=4),
    name='Ground Track',
    hovertemplate='X: %{x:.0f} ft<br>Y: %{y:.0f} ft<extra></extra>'
))

# Direction markers (trace 7)
def get_direction_points(track_x, track_y, num_points=5):
    total_pts = len(track_x)
    indices = np.linspace(total_pts * 0.1, total_pts * 0.85, num_points).astype(int)
    return [track_x[i] for i in indices], [track_y[i] for i in indices]

dir_x, dir_y = get_direction_points(initial_data['track_x'], initial_data['track_y'])
fig.add_trace(go.Scatter(
    x=dir_x, y=dir_y,
    mode='markers',
    marker=dict(size=8, color='white', symbol='circle', line=dict(color='#8e44ad', width=2)),
    name='Waypoints', showlegend=False, hoverinfo='skip'
))

# Base turn marker (trace 8)
fig.add_trace(go.Scatter(
    x=[initial_data['base_turn_point'][0]], y=[initial_data['base_turn_point'][1]],
    mode='markers+text',
    marker=dict(size=12, color='#e67e22', symbol='diamond'),
    text=['Base Turn'], textposition='middle left',
    textfont=dict(size=11, color='#e67e22'),
    name='Base Turn', hoverinfo='skip'
))

# Final turn marker (trace 9)
fig.add_trace(go.Scatter(
    x=[initial_data['final_turn_point'][0]], y=[initial_data['final_turn_point'][1]],
    mode='markers+text',
    marker=dict(size=12, color='#e67e22', symbol='diamond'),
    text=['Final Turn'], textposition='middle right',
    textfont=dict(size=11, color='#e67e22'),
    name='Final Turn', hoverinfo='skip'
))

# Wind vector (trace 10)
fig.add_trace(go.Scatter(
    x=[], y=[],
    mode='lines+markers',
    line=dict(color='#e74c3c', width=3),
    marker=dict(size=[0, 12], symbol=['circle', 'triangle-up']),
    name='Wind', hoverinfo='skip'
))

# Layout with extended Y range
fig.update_layout(
    title=dict(text='Power-Off 180 Ground Track - Runway 29 Left Traffic', font=dict(size=18)),
    xaxis=dict(
        title='Cross-Runway Distance (feet)',
        range=[-7000, 2000],
        zeroline=True, zerolinecolor='lightgray',
        scaleanchor='y', scaleratio=1,
        gridcolor='rgba(200,200,200,0.3)'
    ),
    yaxis=dict(
        title='Along-Runway Distance (feet)',
        range=[-5500, 2000],
        zeroline=True, zerolinecolor='lightgray',
        gridcolor='rgba(200,200,200,0.3)'
    ),
    template='plotly_white',
    height=700, width=950,
    showlegend=True, legend=dict(x=1.02, y=1),
    hovermode='closest'
)

fig.add_annotation(x=0, y=2800, text="29", font=dict(size=24, color='white'), showarrow=False)
fig.add_annotation(x=0, y=250, text="11", font=dict(size=24, color='white'), showarrow=False)

# Altitude profile
fig_alt = go.FigureWidget()

cum_dist, altitude = compute_altitude_profile(initial_data)

fig_alt.add_trace(go.Scatter(
    x=cum_dist, y=altitude,
    mode='lines', line=dict(color='#8e44ad', width=3),
    name='Altitude Profile',
    hovertemplate='Distance: %{x:.0f} ft<br>Altitude: %{y:.0f} ft AGL<extra></extra>'
))

def find_point_index(track_x, track_y, point):
    distances = np.sqrt((track_x - point[0])**2 + (track_y - point[1])**2)
    return np.argmin(distances)

base_idx = find_point_index(initial_data['track_x'], initial_data['track_y'], initial_data['base_turn_point'])
final_idx = find_point_index(initial_data['track_x'], initial_data['track_y'], initial_data['final_turn_point'])
threshold_idx = find_point_index(initial_data['track_x'], initial_data['track_y'], initial_data['threshold_point'])

fig_alt.add_trace(go.Scatter(
    x=[cum_dist[base_idx], cum_dist[final_idx], cum_dist[threshold_idx]],
    y=[altitude[base_idx], altitude[final_idx], altitude[threshold_idx]],
    mode='markers+text',
    marker=dict(size=10, color=['#e67e22', '#e67e22', '#2ecc71'], symbol='diamond'),
    text=['Base Turn', 'Final Turn', 'Threshold'],
    textposition=['top center', 'top center', 'top center'],
    textfont=dict(size=10),
    name='Key Points', hoverinfo='skip'
))

fig_alt.add_hline(y=PATTERN_ALT, line_dash="dash", line_color="gray", 
                  annotation_text="Pattern Alt", annotation_position="right")
fig_alt.add_hline(y=0, line_dash="solid", line_color="green",
                  annotation_text="Touchdown", annotation_position="right")

fig_alt.update_layout(
    title=dict(text=f'Altitude Profile (L/D {GLIDE_RATIO}:1 in air mass)', font=dict(size=16)),
    xaxis=dict(title='Distance Along Track (feet)', gridcolor='rgba(200,200,200,0.3)'),
    yaxis=dict(title='Altitude (ft AGL)', range=[-50, 1100], gridcolor='rgba(200,200,200,0.3)'),
    template='plotly_white',
    height=300, width=950,
    showlegend=False
)


def format_metrics(data):
    alt = data['altitude_budget']
    eff = data['effective_ratios']
    gs = data['ground_speeds']
    legs = data['leg_distances']
    
    return f"""
    <div style="font-family: monospace; padding: 15px; background: #f8f9fa; border-radius: 8px; font-size: 12px;">
        <div style="display: flex; gap: 25px; flex-wrap: wrap;">
            <div>
                <b>Ground Speed</b><br>
                Downwind: {gs['downwind']:.0f} kt<br>
                Base: {gs['base']:.0f} kt<br>
                Final: {gs['final']:.0f} kt
            </div>
            <div>
                <b>Eff. L/D (ground)</b><br>
                Downwind: {eff['downwind']:.1f}:1<br>
                Base: {eff['base']:.1f}:1<br>
                Final: {eff['final']:.1f}:1
            </div>
            <div>
                <b>Leg Distance</b><br>
                Downwind: {legs['downwind']:.0f} ft<br>
                Base: {legs['base']:.0f} ft<br>
                Final: {legs['final']:.0f} ft
            </div>
            <div>
                <b>Altitude Budget</b><br>
                Downwind: {alt['downwind']:.0f} ft<br>
                Base+turns: {alt['base_turn'] + alt['base'] + alt['final_turn']:.0f} ft<br>
                Final: {alt['final']:.0f} ft<br>
                Fixed total: {alt['fixed_total']:.0f} ft<br>
                <b>Total: {alt['total']:.0f} ft</b>
            </div>
            <div>
                <b>Base Turn</b><br>
                {data['dist_past_abeam']:.0f} ft past abeam<br>
                <br>
                <b>Turn Radii</b><br>
                Base: {data['turn_radii']['base']:.0f} ft<br>
                Final: {data['turn_radii']['final']:.0f} ft
            </div>
        </div>
    </div>
    """


def update_plot(change):
    downwind_dist = -downwind_slider.value
    data = compute_power_off_180_track(headwind_slider.value, crosswind_slider.value, downwind_dist)
    
    with fig.batch_update():
        fig.data[5].x = [downwind_dist]
        fig.data[6].x = data['track_x']
        fig.data[6].y = data['track_y']
        
        dir_x, dir_y = get_direction_points(data['track_x'], data['track_y'])
        fig.data[7].x = dir_x
        fig.data[7].y = dir_y
        
        fig.data[8].x = [data['base_turn_point'][0]]
        fig.data[8].y = [data['base_turn_point'][1]]
        fig.data[9].x = [data['final_turn_point'][0]]
        fig.data[9].y = [data['final_turn_point'][1]]
        
        # Wind vector - direction wind is BLOWING
        wind_base_x, wind_base_y = -6500, -5000
        wind_scale = 25
        wind_tip_x = wind_base_x + crosswind_slider.value * wind_scale
        wind_tip_y = wind_base_y - headwind_slider.value * wind_scale
        
        if abs(headwind_slider.value) > 0.1 or abs(crosswind_slider.value) > 0.1:
            fig.data[10].x = [wind_base_x, wind_tip_x]
            fig.data[10].y = [wind_base_y, wind_tip_y]
        else:
            fig.data[10].x = []
            fig.data[10].y = []
    
    cum_dist, altitude = compute_altitude_profile(data)
    base_idx = find_point_index(data['track_x'], data['track_y'], data['base_turn_point'])
    final_idx = find_point_index(data['track_x'], data['track_y'], data['final_turn_point'])
    threshold_idx = find_point_index(data['track_x'], data['track_y'], data['threshold_point'])
    
    with fig_alt.batch_update():
        fig_alt.data[0].x = cum_dist
        fig_alt.data[0].y = altitude
        fig_alt.data[1].x = [cum_dist[base_idx], cum_dist[final_idx], cum_dist[threshold_idx]]
        fig_alt.data[1].y = [altitude[base_idx], altitude[final_idx], altitude[threshold_idx]]
    
    metrics_html.value = format_metrics(data)


headwind_slider.observe(update_plot, names='value')
crosswind_slider.observe(update_plot, names='value')
downwind_slider.observe(update_plot, names='value')
metrics_html.value = format_metrics(initial_data)

display(widgets.VBox([
    widgets.HTML(
        f'<b>Wind Components</b> (direction wind is FROM)<br>'
        f'<i>Headwind +: from ahead on final | Crosswind +: from left of final</i><br>'
        f'<i>Bank: {BANK_ANGLE} deg | Best glide: {GLIDE_AIRSPEED} KIAS | L/D: {GLIDE_RATIO}:1 (air mass)</i>'
    ),
    widgets.HBox([headwind_slider, crosswind_slider]),
    downwind_slider,
    metrics_html
], layout=widgets.Layout(margin='0 0 20px 0')))

display(fig)
display(fig_alt)